<a href="https://colab.research.google.com/github/bamwhamboy/RAG/blob/main/rag_with_chromadb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **RAG: Retrieval Augmented Generation**

### **Using Chroma DB**



In [1]:
pip install -q sentence-transformers requests groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.6 MB/s eta 0:00:00


In [2]:
pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

## **First Step: Ingestion**

Ingestion itself involves multiple steps:

- **Loading** the docuemnts into a readable format
- **Chunking**: Splitting them into smaller chunks
- **Embedding**: Creating their embeddings using the embedding models
- **Indexing**: Storing the embeddings and payloads on the vector DB

### **Loading**

The first step is loading the document so that we can process it.

In [3]:
# Loading the Document
import os
import requests

GITHUB_RAW_URL = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt"

def load_document(url: str) -> str:
    """Fetch a plain-text file from a raw GitHub URL."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.text

raw_text = load_document(GITHUB_RAW_URL)
print(f"Loaded {len(raw_text):,} characters")
print(raw_text[:400])  # Sanity check

Loaded 16,864 characters
# AtliqAI HR Policies

AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.

---

## Employment & Onboarding

### Offer and Joi


## **Chunking**

In [4]:
CHUNK_SIZE = 50

def parse_word_chunks(text: str, chunk_size: int = CHUNK_SIZE) -> list[dict]:
    # Strip markdown heading symbols and blank lines
    clean_lines = []
    for line in text.splitlines():
        line = line.strip().lstrip("#").strip()
        if line:
            clean_lines.append(line)

    # Join everything into one word list and slice
    words = " ".join(clean_lines).split()

    chunks = []
    for i in range(0, len(words), chunk_size):
        content = " ".join(words[i : i + chunk_size])
        chunks.append({
            "chunk_index": len(chunks),
            "content": content,
        })

    return chunks

In [5]:
chunks = parse_word_chunks(raw_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 51


In [6]:
# Inspect a chunk
for chunk in chunks[:3]:
    print("─" * 55)
    print(f"Content : {chunk['content'][:200]}…")

───────────────────────────────────────────────────────
Content : AtliqAI HR Policies AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compe…
───────────────────────────────────────────────────────
Content : & Onboarding Offer and Joining Formalities Upon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team will shar…
───────────────────────────────────────────────────────
Content : photograph. Failure to submit required documents within 7 working days of joining may result in withholding of the first salary disbursement. Probation Period All new employees at AtliqAI are placed o…


In [7]:
def build_chunk_text(chunk: dict) -> str:
    return chunk["content"]

# **Embedding**

In [8]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
# Extract Chunk Texts
chunk_texts = [build_chunk_text(c) for c in chunks]

print(f"Embedding {len(chunk_texts)} chunks …")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")

Embedding 51 chunks …


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Shape: (51, 384)


## **Indexing**

In [10]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize ChromaDB client
# For persistent storage, you can specify a path:
# client = chromadb.PersistentClient(path="/tmp/chromadb_data")
# For in-memory, use:
client = chromadb.Client()

# Define the embedding function for ChromaDB to use the same SentenceTransformer model
chroma_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)

COLLECTION_NAME = "docs"

# Get or create the collection. If it exists, it will be loaded; otherwise, it will be created.
# ChromaDB handles collection existence automatically with get_or_create_collection
collection = client.get_or_create_collection(name=COLLECTION_NAME, embedding_function=chroma_ef)

print(f"Collection '{COLLECTION_NAME}' ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection 'docs' ready.


In [11]:
# Preparing data for ChromaDB
# ChromaDB's add method expects separate lists for ids, embeddings, metadatas, and documents
ids = [str(chunk['chunk_index']) for chunk in chunks]
metadatas = [{
    "content": chunk["content"],
    # Add any other metadata you want to store and filter by
} for chunk in chunks]
documents = [chunk["content"] for chunk in chunks] # ChromaDB can store documents directly

# Add the data to the collection
# We don't need to pass embeddings here if we set an embedding_function on the collection
collection.add(
    embeddings=embeddings.tolist(), # Pass the pre-computed embeddings
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Indexed {len(ids)} documents in ChromaDB collection '{COLLECTION_NAME}'.")

Indexed 51 documents in ChromaDB collection 'docs'.


In [12]:
# ChromaDB doesn't have a direct equivalent to `get_collection` info that shows vector size easily after adding embeddings.
# However, we can check the count of items in the collection.
print(f"Points     : {collection.count()}")
# The dimension is determined by the embedding model, which is 384.
print(f"Dimensions : {embedder.get_sentence_embedding_dimension()}")

Points     : 51
Dimensions : 384


/tmp/ipykernel_715/1096598512.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimensions : {embedder.get_sentence_embedding_dimension()}")


# **Retrieval**

In [13]:
def retrieve(query: str, top_k: int = 5) -> list[dict]:
    """
    Embed the query and return the top-k most similar chunks using ChromaDB.

    Args:
        query          : User's question.
        top_k          : Number of chunks to return.
    """
    query_embedding = embedder.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=['metadatas', 'distances'] # Request metadatas and distances
    )

    retrieved_chunks = []
    if results and results['metadatas']:
        # Unpack results. ChromaDB returns lists of lists if multiple query embeddings are provided.
        # We only provided one, so we take the first item.
        for i in range(len(results['metadatas'][0])):
            metadata = results['metadatas'][0][i]
            distance = results['distances'][0][i]
            retrieved_chunks.append({
                "content": metadata["content"],
                "score": 1 - distance # Cosine similarity score (1 - distance) for better interpretation
            })
    return retrieved_chunks

In [14]:
results = retrieve("What is the leave policy")
for r in results:
    print(f"[score={r['score']}]")
    print(f"  {r['content'][:200]}…\n")

[score=0.5587958693504333]
  Policy Casual Leave Every confirmed employee is entitled to 12 casual leaves per calendar year, credited at 1 leave per month. Casual leave can be availed for personal errands, minor illness, or unpla…

[score=0.46427613496780396]
  of more than 2 consecutive days. Unused sick leaves up to a maximum of 10 can be carried forward to the following year. Earned Leave Employees accrue earned leave at the rate of 1.25 days per month, a…

[score=0.4213528633117676]
  earned leaves (up to the carry-forward limit), reimbursement of pending expense claims, and deduction of any outstanding dues or advances. The relieving letter and experience certificate will be issue…

[score=0.4150383472442627]
  be carried forward to the next calendar year and lapse on December 31st. Sick Leave Employees are entitled to 10 sick leaves per calendar year. Sick leave can be availed in case of illness, hospitalis…

[score=0.40104150772094727]
  HR team. The purpose of the exit intervie

### **RAG Pipeline**

In [15]:
SYSTEM_PROMPT = """You are a helpful HR assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information."""

In [16]:
def build_context(retrieved_chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(f"[Source {i}]\n{chunk['content']}")
    return "\n\n---\n\n".join(parts)

In [17]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


### **End-to-end RAG pipeline:**
  1. Retrieve relevant chunks
  2. Format them as a context block
  3. Send context + query to Groq and return the answer

In [18]:
from groq import Groq

groq_client = Groq()   # Reads GROQ_API_KEY from environment automatically
GROQ_MODEL  = "openai/gpt-oss-safeguard-20b"

def rag(query: str, top_k: int = 5):
    """
    End-to-end RAG pipeline:
      1. Retrieve relevant chunks
      2. Format them as a context block
      3. Send context + query to Groq and return the answer
    """
    # Step 1 — Retrieve
    chunks = retrieve(query, top_k=top_k)
    if not chunks:
        return "No relevant content found in the document."

    # Step 2 — Build context
    context = build_context(chunks)

    # Step 3 — Generate
    user_message = f"Context:\n{context}\n\nQuestion: {query}"

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.2,   # Low = factual;  High = creative
    )
    return response.choices[0].message.content, context

In [19]:
answer, context = rag("What are the main topics covered in this document?")
print(answer)
print(f"{250*'='}")
print(f"\n\nSOURCES:\n {context}")

The document covers several key HR‑related areas, including:

1. **Employment** – general policies on hiring, conduct, compensation, and well‑being (Source 1).  
2. **Data Privacy & Confidentiality** – rules for handling employee personal data and the role of the Head of HR in policy interpretation (Source 2).  
3. **Onboarding** – procedures and required documentation after an offer is accepted (Source 3).  
4. **Health Insurance** – coverage details for employees and dependents, including waiting periods and optional enhancements (Source 4).  
5. **Exit Interview** – purpose and voluntary nature of the interview to gather feedback and aid transition (Source 5).  
6. **Grievance Redressal** – process for raising work‑related concerns (Source 5).


SOURCES:
 [Source 1]
AtliqAI HR Policies AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-be